In [1]:
import os
import pickle

In [2]:
explanations_dir = '/home/ubuntu/Multimodal-Uncertainty-Quantification/runs/llava_gqa_yes_gsam_grounding/explanations'
grounding_dir = '/home/ubuntu/Multimodal-Uncertainty-Quantification/runs/llava_gqa_yes_gsam_grounding/grounding'

In [3]:
exp_files = os.listdir(explanations_dir)
ground_files = os.listdir(grounding_dir)

exp_file_temp_path = exp_files[0]
ground_file_temp_path = ground_files[0]

In [4]:
# Check Grounding Path
grounding_path = os.path.join(grounding_dir, ground_file_temp_path)
with open(grounding_path, 'rb') as f:
    grounding = pickle.load(f)

/opt/conda/envs/llava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2024-10-09 17:16:31,577] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [6]:
grounding['response_0']

dict_keys(['prompt', 'outputs', 'generated_token_ids', 'decoded_outputs', 'error_detection'])

In [6]:
with open(os.path.join(explanations_dir, exp_file_temp_path), 'rb') as f:
    exp_temp = pickle.load(f)

/opt/conda/envs/llava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2024-09-20 17:32:07,453] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [9]:
resp_0_output = exp_temp['response_0']['outputs']
resp_0_output.keys()

odict_keys(['sequences', 'scores'])

In [10]:
resp_0_output['sequences'].shape

torch.Size([1, 104])

In [14]:
len(resp_0_output['scores'])

41

In [16]:
from transformers import AutoProcessor
llava_processor = AutoProcessor.from_pretrained('llava-hf/llava-1.5-7b-hf')



In [24]:
generated_tokens = resp_0_output['sequences'][0][-41:]
llava_processor.decode(generated_tokens)

'\n        {\n          "answer": false,\n          "explanation": "The bus is in front of the person",\n          "confidence": 0.9\n        }</s>'

In [23]:
generated_tokens.shape

torch.Size([63])

In [27]:
import torch
torch.stack(resp_0_output['scores']).squeeze().shape

torch.Size([41, 32064])

In [7]:
path = '/home/ubuntu/Multimodal-Uncertainty-Quantification/runs/llava_gqa_yes_gsam_grounding_random_100/grounding/grounding_0_0_181060668.pkl'
with open(path, 'rb') as f:
    grounding = pickle.load(f)

In [15]:
grounding['response_2'].keys()

dict_keys(['prompt', 'outputs', 'generated_token_ids', 'decoded_outputs', 'detections', 'image', 'grounding_score'])

In [1]:
path = '/home/ubuntu/Multimodal-Uncertainty-Quantification/runs/llava_gqa_yes_gsam_grounding_random_100/grounding_test/grounding_077759.pkl'
import pickle
with open(path, 'rb') as f:
    grounding = pickle.load(f)

/opt/conda/envs/llava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2024-10-12 08:48:18,422] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [12]:
# grounding['response_2']['detections'][0]
grounding['response_2'].keys()

dict_keys(['decoded_outputs', 'detections', 'grounding_score'])

In [35]:
path = '/home/ubuntu/Multimodal-Uncertainty-Quantification/runs/llava_gqa_yes_gsam_grounding_random_100/explanations_test/explanations_077759.pkl'


In [36]:
exp.keys()

dict_keys(['question_id', 'response_0', 'response_1', 'response_2', 'response_3', 'response_4', 'response_5', 'response_6', 'response_7', 'response_8', 'response_9', 'response_10', 'response_11', 'response_12', 'response_13', 'response_14', 'response_15', 'response_16', 'response_17', 'response_18', 'response_19'])

In [37]:
exp['response_2'].keys()

dict_keys(['decoded_outputs', 'detections', 'grounding_score'])

In [39]:
exp['response_2']['grounding_score']

0.305490106344223

In [28]:
exp['response_2']['transition_scores'].shape

(1, 47)

In [22]:
import numpy as np
import torch
import h5py
# Function to store a dictionary into an HDF5 file
#  Recursive function to save nested dictionaries in HDF5
def save_dict_to_hdf5(group, data_dict):
    for key, value in data_dict.items():
        if isinstance(value, str):
            group.create_dataset(key, data=value)
        elif isinstance(value, list) and all(isinstance(i, str) for i in value):
            dt = h5py.special_dtype(vlen=str)
            group.create_dataset(key, data=value, dtype=dt)
        elif isinstance(value, torch.Tensor):
            group.create_dataset(key, data=value.numpy())
        elif isinstance(value, np.ndarray):
            group.create_dataset(key, data=value)
        elif isinstance(value, dict):
            # Recursively save nested dictionaries
            subgroup = group.create_group(key)
            save_dict_to_hdf5(subgroup, value)
        else:
            raise TypeError(f"Unsupported data type for key {key}: {type(value)}")

# Call the function to save the data dictionary to HDF5 file
with h5py.File('/home/ubuntu/Multimodal-Uncertainty-Quantification/runs/llava_gqa_yes_gsam_grounding_random_100/explanations_test/data.h5', 'w') as f:
    save_dict_to_hdf5(f, exp)

# # Call the function to save the data dictionary
# save_dict_to_hdf5(exp, '/home/ubuntu/Multimodal-Uncertainty-Quantification/runs/llava_gqa_yes_gsam_grounding_random_100/explanations_test/data.h5')

# # Loading the data back from HDF5 for verification
# with h5py.File('data.h5', 'r') as f:
#     for key in f.keys():
#         print(f"{key}: {f[key][()]}")

TypeError: can't convert cuda:2 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [19]:
import h5py
import numpy as np

# # Create a dictionary with mixed data
# data = {
#     'text': 'The quick brown fox',
#     'tensor': np.random.rand(100, 100)
# }

# Save the dictionary
with h5py.File('data.h5', 'w') as f:
    f.create_dataset('tensor', data=data['tensor'])
    f.attrs['text'] = data['text']  # Store string as an attribute

NameError: name 'data' is not defined

In [23]:
with open('/home/ubuntu/Multimodal-Uncertainty-Quantification/runs/llava_gqa_yes_gsam_grounding_random_100/explanations_test/data.pkl', 'wb') as f:
    pickle.dump(exp, f, protocol=pickle.HIGHEST_PROTOCOL)